### Python packages used in this code

In [1]:
import importlib
import sys
utils_path = './'
data_path = './data'
if utils_path not in sys.path:
    sys.path.append(utils_path)

from MyPackages import *
from MyFunctions import *
from MyTorch import *
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms = True
MyDevice = set_device('')

%matplotlib inline
MySEED = 373



def fun_Reload(pkg):
    module = importlib.import_module(pkg)
    module = importlib.reload(module)
    attrs = {attr: getattr(module, attr) for attr in dir(module)}
    globals().update(attrs)
    
fun_Reload('MyPackages')
fun_Reload('MyFunctions')
fun_Reload('MyTorch')


print('--Platform--')
print('OS :', platform.platform())
print('--Version--')
print('python : ', sys.version)
print('numpy :', np.__version__)
print('pandas :', pd.__version__)
print('sklearn :', pd.__version__)
print('PyTorch :', torch.__version__)

cpu
--Platform--
OS : Linux-6.5.11-linuxkit-x86_64-with-glibc2.31
--Version--
python :  3.9.7 | packaged by conda-forge | (default, Sep 29 2021, 19:23:11) 
[GCC 9.4.0]
numpy : 1.26.4
pandas : 2.0.3
sklearn : 2.0.3
PyTorch : 1.12.1


## Environments
--Platform--

OS : Linux-6.5.11-linuxkit-x86_64-with-glibc2.31

--Version--

python :  3.9.7 | packaged by conda-forge | (default, Sep 29 2021, 19:23:11) 

[GCC 9.4.0]

numpy : 1.26.4

pandas : 2.0.3

sklearn : 2.0.3

PyTorch : 1.12.1

# Check scaling law
# 1. Load data

In [10]:
with open(f'{data_path}/Dataset_source.pkl', 'rb') as f:
    data_dict_source = pickle.load(f)
    
with open(f'{data_path}/Dataset_target.pkl', 'rb') as f:
    data_dict_target = pickle.load(f)
    
    
## ----------------------------------------------------------------
## Dataset - source
x_source = data_dict_source['Inputs']
y_source_all = data_dict_source['Outputs']
Scaler_source_all = data_dict_source['Scaler_outputs']

# 2. Setups
- 200 loops for different seeds
- 200 loops for different $N_s$

In [11]:
## Iteration setting
n_seed_list = np.arange(3)
argname_list = data_dict_source['Orginal_outputs'].columns.to_list()

argname_list = data_dict_source['Orginal_outputs'].columns.to_list()

IMD_idx = data_dict_target['IMD_idx']

# 3. Main
#### About functions used
- In `MyTorch.py`
    - `MyModel` : Neural network module. Initialize the weights and biases via `model.initialize_weights()`
    - `fun_CreateDataloader()` : Create Dataloader object for mini-batch training
    - `fun_MakePrediction()` : Make prediction, return pandas dataframe with "Scaled Observation" and "Scaled Prediction"
- In `MyFunctions.py`
    - `plot_scatter()` : Draw a scatter plot
    - `fun_PltScaling` : Draw a scaling curve
    - `MyTime()` : Estimate the remaining execution time

## Source

In [ ]:
%%time
start_time = datetime.datetime.now()


i=0
for argname in argname_list:
    

    ## ----------------------------------------------------------------
    ## Dataset - source
    y_source = y_source_all[argname]
    idx_source = y_source.index.tolist()
    Scaler_source = Scaler_source_all[argname]

    
    # Loop combination
    n_samples_source_list = [0]+(np.floor(10**np.linspace(2, np.log10(len(idx_source)), 10))).astype(np.int32).tolist()
    
    combinations = list(itertools.product(
        n_seed_list,
        n_samples_source_list
    ))
    
    ## ----------------------------------------------------------------
    ## Directories
    result_path = f'./Result/{argname}'
    model_path = f'./Result/{argname}/Source_Model'
    fun_MakeDir([result_path, model_path])
    
    for n_seed, n_samples_source in combinations:
        print(argname)

        loop_id = f'Seed{n_seed}NumSource{n_samples_source}'
        print(loop_id)

        ## ----------------------------------------------------------------
        ## Model definition
        fix_seed(n_seed)
        model = MyModel(
            input_dim = x_target.shape[1], 
            layers_dict = {
                'fc1': 512, 
                'fc2': 256,
                'fc3': 128,
                'fc4': 64,
                'fc5': 32,
                'fc6': 1
            }
        )
        model.initialize_weights(seed=n_seed)
        model.to(MyDevice)

        ### Training parameters
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam
        learning_rate_source = 1e-3
        learning_rate_target = {
            'fc1' : 1e-3,
            'fc2' : 1e-3,
            'fc3' : 1e-3,
            'fc4' : 1e-3,
            'fc5' : 1e-3,
            'fc6' : 1e-3
        }
        n_epochs = 1000


        ### Early stopping parameters
        patience = 5
        loss_th_source = None
        loss_th_target = None
        best_loss = None
        epochs_no_improve = 0



        ## ----------------------------------------------------------------
        ## Data splitting
        fix_seed(n_seed)
        print(' ')
        ### Source
        if n_samples_source > 0:
            shuffle_idx_source = idx_source.copy()
            fix_seed(n_seed)
            np.random.shuffle(shuffle_idx_source)
            idx_trainval_source = shuffle_idx_source[:n_samples_source]
            idx_train_source = idx_trainval_source[:int(n_samples_source*0.9)]
            idx_val_source = idx_trainval_source[int(n_samples_source*0.9):]
        else:
            idx_train_source, idx_val_source = [], []
            
        for idx_name in ['idx_train_source', 'idx_val_source']:
            globals()[idx_name] = pd.Series(globals()[idx_name])


        ## ----------------------------------------------------------------
        ## Create dataset and dataloader
        ### Dataset
        if n_samples_source > 0:
            dataset_train_source = fun_CreateDataset(
                x_source,
                idx_train_source,
                y_source,
                MyDevice
            )
            dataset_val_source = fun_CreateDataset(
                x_source,
                idx_val_source,
                y_source,
                MyDevice
            )

        ### Dataloader
        if n_samples_source > 0:
            dataloader_train_source = DataLoader(
                MyDataset(*dataset_train_source),
                batch_size=32, 
                shuffle=False
            )


        if n_samples_source > 0:
            ## ----------------------------------------------------------------
            ## Training - source
            model, train_losses_source, val_losses_source = fun_TrainModel(
                model = model, 
                dataloader_train = dataloader_train_source, 
                dataset_val = dataset_val_source, 
                learning_rate = learning_rate_source,
                n_epochs = n_epochs, 
                patience = patience,
                loss_th = loss_th_source,
                opt = optimizer,
                seed = n_seed,
                device=MyDevice,
                weight_decay=0,
                lr_decay=0,
                amsgrad=True,
                schedule=False
            )
            torch.save(model.state_dict(), f'{model_path}/SourceModelWeight_{loop_id}.pth')

density
Seed0NumSource0
 
density
Seed0NumSource100
 
Epoch 1: train loss = 0.8897393544514974, val loss = 0.6612810492515564
Epoch 10: train loss = 0.010183105555673441, val loss = 0.2615097463130951
Early stopping!
density
Seed0NumSource205
 
Epoch 1: train loss = 0.7544958641131719, val loss = 0.809790849685669
Epoch 10: train loss = 0.011597080156207085, val loss = 0.5047215819358826
Early stopping!
density
Seed0NumSource420
 
Epoch 1: train loss = 0.6664760534962019, val loss = 0.5328337550163269
Epoch 10: train loss = 0.09179865568876266, val loss = 0.21466445922851562
Early stopping!
density
Seed0NumSource861
 
Epoch 1: train loss = 0.5102468127012253, val loss = 0.18896256387233734
Epoch 10: train loss = 0.06241876132786274, val loss = 0.11662936955690384
Early stopping!
density
Seed0NumSource1766
 
Epoch 1: train loss = 0.3359321001172066, val loss = 0.15781816840171814
Epoch 10: train loss = 0.043345379047095775, val loss = 0.10691981017589569
Early stopping!
density
Seed0Num

## Tm

In [12]:
%%time
start_time = datetime.datetime.now()

df_results = pd.DataFrame({
        'Property' : [],
        'Seed' : [],
        'NumSourceSamples' : [],
        'Tm_RMSE' : [],
        'Tm_MAE' : [],
        'Tm_IMD_RMSE' : [],
        'Tm_IMD_MAE' : [],  
    })

i=0
for argname in argname_list:
    
    pkl_dict = {} # For save
    
    ## ----------------------------------------------------------------
    ## Dataset - source
    y_source = y_source_all[argname]
    idx_source = y_source.index.tolist()
    Scaler_source = Scaler_source_all[argname]

    y_target = data_dict_target['Outputs_Tm']
    x_target = data_dict_target['Inputs']
    Scaler_target = data_dict_target['Scaler_outputs_Tm']
    idx_target = y_target.index.tolist()
    
    
    # Loop combination
    n_samples_source_list = [0]+(np.floor(10**np.linspace(2, np.log10(len(idx_source)), 10))).astype(np.int32).tolist()

    combinations = list(itertools.product(
        n_seed_list,
        n_samples_source_list
    ))
    
    ## ----------------------------------------------------------------
    ## Directories
    result_path = f'./Result/{argname}'
    model_path = f'./Result/{argname}/Target_Tm_Model'
    fun_MakeDir([result_path, model_path])
    
    for n_seed, n_samples_source in combinations:
        print(argname)

        loop_id = f'Seed{n_seed}NumSource{n_samples_source}'
        print(loop_id)
        
        ## ----------------------------------------------------------------
        ## Model definition
        fix_seed(n_seed)
        model = MyModel(
            input_dim = x_target.shape[1], 
            layers_dict = {
                'fc1': 512, 
                'fc2': 256,
                'fc3': 128,
                'fc4': 64,
                'fc5': 32,
                'fc6': 1
            }
        )
        model.initialize_weights(seed=n_seed)
        model.to(MyDevice)

        ### Training parameters
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam
        learning_rate_source = 1e-3
        learning_rate_target = {
            'fc1' : 1e-3,
            'fc2' : 1e-3,
            'fc3' : 1e-3,
            'fc4' : 1e-3,
            'fc5' : 1e-3,
            'fc6' : 1e-3
        }
        n_epochs = 1000


        ### Early stopping parameters
        patience = 30
        loss_th_source = None
        loss_th_target = None
        best_loss = None
        epochs_no_improve = 0
        
        fix_seed(n_seed)
        ### Target
        idx_trainvalcal_target, idx_test_target = train_test_split(idx_target, random_state=n_seed, test_size=0.2)
        idx_train_target, idx_valcal_target = train_test_split(idx_trainvalcal_target, random_state=n_seed, test_size=2/8)
        idx_val_target, idx_cal_target = train_test_split(idx_valcal_target, random_state=n_seed, test_size=1/2)
        
        for idx_name in ['idx_train_target', 'idx_val_target', 'idx_cal_target', 'idx_test_target']:
            globals()[idx_name] = pd.Series(globals()[idx_name])

        dataset_train_target = fun_CreateDataset(
            x_target,
            idx_train_target,
            y_target,
            MyDevice
        )
        dataset_val_target = fun_CreateDataset(
            x_target,
            idx_val_target,
            y_target,
            MyDevice
        )
        dataset_cal_target = fun_CreateDataset(
            x_target,
            idx_cal_target,
            y_target,
            MyDevice
        )
        dataset_test_target = fun_CreateDataset(
            x_target,
            idx_test_target,
            y_target,
            MyDevice
        )
        
        dataloader_train_target = DataLoader(
            MyDataset(*dataset_train_target),
            batch_size=32,
            shuffle=False
        )
        if n_samples_source > 0:
            ## ----------------------------------------------------------------
            model.load_state_dict(torch.load(f'./Result/{argname}/Source_Model/SourceModelWeight_{loop_id}.pth'))
        df_fits_source = None
            
        ## Training - target
        model, train_losses_target, val_losses_target = fun_TrainModel(
            model = model, 
            dataloader_train = dataloader_train_target,
            dataset_val = dataset_val_target, 
            learning_rate = learning_rate_source,
            n_epochs = n_epochs, 
            patience = patience,
            loss_th = loss_th_target,
            opt = optimizer,
            seed = n_seed,
            device=MyDevice,
            weight_decay=0,
            lr_decay=0,
            amsgrad=True,
            schedule=False
        )
        torch.save(model.state_dict(), f'{model_path}/TargetModelWeight_{loop_id}.pth')

        model.eval()
        with torch.no_grad():
            df_fits_target = fun_MakePrediction(model, dataset_train_target, idx_train_target, True, MyDevice)
            df_pred_target = fun_MakePrediction(model, dataset_test_target, idx_test_target, True, MyDevice)
            df_pred_cal_target = fun_MakePrediction(model, dataset_cal_target, idx_cal_target, True, MyDevice)

        test_idx_Tm_IMD = list(set(idx_test_target.to_list()) & set(IMD_idx))

        ### Scaling
        df_pred_target['Observation'] = Scaler_target.inverse(df_pred_target['Scaled Observation'].rename('Tm'))
        df_pred_target['Prediction'] = Scaler_target.inverse(df_pred_target['Scaled Prediction'].rename('Tm'))
        df_pred_cal_target['Observation'] = Scaler_target.inverse(df_pred_cal_target['Scaled Observation'].rename('Tm'))
        df_pred_cal_target['Prediction'] = Scaler_target.inverse(df_pred_cal_target['Scaled Prediction'].rename('Tm'))
        
        if len(test_idx_Tm_IMD) > 0:
            IMD_RMSE = mean_squared_error(df_pred_target['Observation'].loc[test_idx_Tm_IMD], df_pred_target['Prediction'].loc[test_idx_Tm_IMD], squared=False)
            IMD_MAE = mean_absolute_error(df_pred_target['Observation'].loc[test_idx_Tm_IMD], df_pred_target['Prediction'].loc[test_idx_Tm_IMD])
        else:
            IMD_RMSE = np.nan
            IMD_MAE = np.nan
            
        df_results.loc[len(df_results)] = [argname, n_seed,
                                           n_samples_source, 
                                           mean_squared_error(df_pred_target['Observation'], df_pred_target['Prediction'], squared=False), 
                                           mean_absolute_error(df_pred_target['Observation'], df_pred_target['Prediction']),
                                           IMD_RMSE,
                                           IMD_MAE,
                                           ]
        df_results.to_csv(f'./Result/Tm_scale.csv')
        
        pkl_dict[f'{loop_id}'] = {
            'train_idx': idx_train_target.to_list(),
            'cal_idx':idx_cal_target.to_list(),
            'test_idx': idx_test_target.to_list(),
            'train_x':x_target.loc[idx_train_target],
            'train_y':y_target.loc[idx_train_target],
            'cal_x': x_target.loc[idx_cal_target],
            'cal_y': df_pred_cal_target['Observation'],
            'cal_pred_y': df_pred_cal_target['Prediction'],
            'model': model.state_dict(),
            'scaler_x':data_dict_target['Scaler_inputs'],
            'scaler_y':Scaler_target
        }
        with open(f'{model_path}/all.pickle', 'wb') as f:
            pickle.dump(pkl_dict, f)
        f.close()
        

density
Seed0NumSource0
Epoch 1: train loss = 2.7874963680903115, val loss = 0.8103026747703552
Epoch 10: train loss = 0.06566069779607157, val loss = 1.3902653455734253
Epoch 20: train loss = 0.022823816902625065, val loss = 1.6335911750793457
Epoch 30: train loss = 0.07133539393544197, val loss = 1.6345630884170532
Early stopping!
density
Seed0NumSource100
Epoch 1: train loss = 1.2894810040791829, val loss = 2.7851505279541016
Epoch 10: train loss = 0.06338671408593655, val loss = 1.4537521600723267
Epoch 20: train loss = 0.02211563941091299, val loss = 1.2137751579284668
Epoch 30: train loss = 0.01132083140934507, val loss = 1.2551382780075073
Epoch 40: train loss = 0.015314595773816109, val loss = 1.386745572090149
Early stopping!
density
Seed0NumSource205
Epoch 1: train loss = 1.427536129951477, val loss = 1.219905138015747
Epoch 10: train loss = 0.05885726911947131, val loss = 2.2318925857543945
Epoch 20: train loss = 0.048271955301364265, val loss = 2.2001729011535645
Epoch 30: 

## Tiso

In [13]:
%%time
start_time = datetime.datetime.now()

df_results = pd.DataFrame({
        'Property' : [],
        'Seed' : [],
        'NumSourceSamples' : [],
        'Tiso_RMSE' : [],
        'Tiso_MAE' : [],
        'Tiso_IMD_RMSE' : [],
        'Tiso_IMD_MAE' : [],  
    })

i=0
for argname in argname_list:
    
    pkl_dict = {} # For save
    ## ----------------------------------------------------------------
    ## Dataset - source
    y_source = y_source_all[argname]
    idx_source = y_source.index.tolist()
    Scaler_source = Scaler_source_all[argname]

    y_target = data_dict_target['Outputs_Tiso']
    x_target = data_dict_target['Inputs']
    Scaler_target = data_dict_target['Scaler_outputs_Tiso']
    idx_target = y_target.index.tolist()
    
    
    # Loop combination
    n_samples_source_list = [0]+(np.floor(10**np.linspace(2, np.log10(len(idx_source)), 10))).astype(np.int32).tolist()

    combinations = list(itertools.product(
        n_seed_list,
        n_samples_source_list
    ))
    
    ## ----------------------------------------------------------------
    ## Directories
    result_path = f'./Result/{argname}'
    model_path = f'./Result/{argname}/Target_Tiso_Model'
    fun_MakeDir([result_path, model_path])
    
    for n_seed, n_samples_source in combinations:
        print(argname)

        loop_id = f'Seed{n_seed}NumSource{n_samples_source}'
        print(loop_id)
        
        ## ----------------------------------------------------------------
        ## Model definition
        fix_seed(n_seed)
        model = MyModel(
            input_dim = x_target.shape[1], 
            layers_dict = {
                'fc1': 512, 
                'fc2': 256,
                'fc3': 128,
                'fc4': 64,
                'fc5': 32,
                'fc6': 1
            }
        )
        model.initialize_weights(seed=n_seed)
        model.to(MyDevice)

        ### Training parameters
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam
        learning_rate_source = 1e-3
        learning_rate_target = {
            'fc1' : 1e-3,
            'fc2' : 1e-3,
            'fc3' : 1e-3,
            'fc4' : 1e-3,
            'fc5' : 1e-3,
            'fc6' : 1e-3
        }
        n_epochs = 1000


        ### Early stopping parameters
        patience = 30
        loss_th_source = None
        loss_th_target = None
        best_loss = None
        epochs_no_improve = 0
        
        fix_seed(n_seed)
        ### Target
        idx_trainvalcal_target, idx_test_target = train_test_split(idx_target, random_state=n_seed, test_size=0.2)
        idx_train_target, idx_valcal_target = train_test_split(idx_trainvalcal_target, random_state=n_seed, test_size=2/8)
        idx_val_target, idx_cal_target = train_test_split(idx_valcal_target, random_state=n_seed, test_size=1/2)
        
        for idx_name in ['idx_train_target', 'idx_val_target', 'idx_cal_target', 'idx_test_target']:
            globals()[idx_name] = pd.Series(globals()[idx_name])

        dataset_train_target = fun_CreateDataset(
            x_target,
            idx_train_target,
            y_target,
            MyDevice
        )
        dataset_val_target = fun_CreateDataset(
            x_target,
            idx_val_target,
            y_target,
            MyDevice
        )
        dataset_cal_target = fun_CreateDataset(
            x_target,
            idx_cal_target,
            y_target,
            MyDevice
        )
        dataset_test_target = fun_CreateDataset(
            x_target,
            idx_test_target,
            y_target,
            MyDevice
        )
        
        dataloader_train_target = DataLoader(
            MyDataset(*dataset_train_target),
            batch_size=32,
            shuffle=False
        )
        if n_samples_source > 0:
            ## ----------------------------------------------------------------
            model.load_state_dict(torch.load(f'./Result/{argname}/Source_Model/SourceModelWeight_{loop_id}.pth'))
            
        ## Training - target
        model, train_losses_target, val_losses_target = fun_TrainModel(
            model = model, 
            dataloader_train = dataloader_train_target,
            dataset_val = dataset_val_target, 
            learning_rate = learning_rate_source,
            n_epochs = n_epochs, 
            patience = patience,
            loss_th = loss_th_target,
            opt = optimizer,
            seed = n_seed,
            device=MyDevice,
            weight_decay=0,
            lr_decay=0,
            amsgrad=True,
            schedule=False
        )
        torch.save(model.state_dict(), f'{model_path}/TargetModelWeight_{loop_id}.pth')

        model.eval()
        with torch.no_grad():
            df_fits_target = fun_MakePrediction(model, dataset_train_target, idx_train_target, True, MyDevice)
            df_pred_target = fun_MakePrediction(model, dataset_test_target, idx_test_target, True, MyDevice)
            df_pred_cal_target = fun_MakePrediction(model, dataset_cal_target, idx_cal_target, True, MyDevice)


        test_idx_Tm_IMD = list(set(idx_test_target.to_list()) & set(IMD_idx))

        ### Scaling
        df_pred_target['Observation'] = Scaler_target.inverse(df_pred_target['Scaled Observation'].rename('Tiso'))
        df_pred_target['Prediction'] = Scaler_target.inverse(df_pred_target['Scaled Prediction'].rename('Tiso'))
        df_pred_cal_target['Observation'] = Scaler_target.inverse(df_pred_cal_target['Scaled Observation'].rename('Tiso'))
        df_pred_cal_target['Prediction'] = Scaler_target.inverse(df_pred_cal_target['Scaled Prediction'].rename('Tiso'))
        
        
        
        if len(test_idx_Tm_IMD) > 0:
            IMD_RMSE = mean_squared_error(df_pred_target['Observation'].loc[test_idx_Tm_IMD], df_pred_target['Prediction'].loc[test_idx_Tm_IMD], squared=False)
            IMD_MAE = mean_absolute_error(df_pred_target['Observation'].loc[test_idx_Tm_IMD], df_pred_target['Prediction'].loc[test_idx_Tm_IMD])
        else:
            IMD_RMSE = np.nan
            IMD_MAE = np.nan
            
        df_results.loc[len(df_results)] = [argname, n_seed,
                                           n_samples_source, 
                                           mean_squared_error(df_pred_target['Observation'], df_pred_target['Prediction'], squared=False), 
                                           mean_absolute_error(df_pred_target['Observation'], df_pred_target['Prediction']),
                                           IMD_RMSE,
                                           IMD_MAE,
                                           ]
        df_results.to_csv(f'./Result/Tiso_scale.csv')
        pkl_dict[f'{loop_id}'] = {
            'train_idx': idx_train_target.to_list(),
            'cal_idx':idx_cal_target.to_list(),
            'test_idx': idx_test_target.to_list(),
            'train_x':x_target.loc[idx_train_target],
            'train_y':y_target.loc[idx_train_target],
            'cal_x': x_target.loc[idx_cal_target],
            'cal_y': df_pred_cal_target['Observation'],
            'cal_pred_y': df_pred_cal_target['Prediction'],
            'model': model.state_dict(),
            'scaler_x':data_dict_target['Scaler_inputs'],
            'scaler_y':Scaler_target
        }
        with open(f'{model_path}/all.pickle', 'wb') as f:
            pickle.dump(pkl_dict, f)
        f.close()

density
Seed0NumSource0
Epoch 1: train loss = 1.7362691561381023, val loss = 2.0499818325042725
Epoch 10: train loss = 0.15713106965025267, val loss = 0.5689369440078735
Epoch 20: train loss = 0.07178087594608466, val loss = 0.5250489115715027
Epoch 30: train loss = 0.0640951016296943, val loss = 0.5232822895050049
Epoch 40: train loss = 0.040232677633563675, val loss = 0.5056606531143188
Epoch 50: train loss = 0.018546932997802894, val loss = 0.4880256652832031
Epoch 60: train loss = 0.009038834366947412, val loss = 0.47665172815322876
Epoch 70: train loss = 0.025589533150196075, val loss = 0.4432603418827057
Epoch 80: train loss = 0.02759549580514431, val loss = 0.46675822138786316
Epoch 90: train loss = 0.0039389893257369595, val loss = 0.47543075680732727
Epoch 100: train loss = 0.0017420388758182526, val loss = 0.47979310154914856
Early stopping!
density
Seed0NumSource100
Epoch 1: train loss = 1.2201960384845734, val loss = 0.636940598487854
Epoch 10: train loss = 0.11951425174872

## LC Classification

In [17]:

%%time
start_time = datetime.datetime.now()

df_results = pd.DataFrame({
        'Property' : [],
        'Seed' : [],
        'NumSourceSamples' : [],
        'N_precision' : [],
        'N_recall' : [],
        'N_f1-score' : [],
        'N_support' : [],
        'S_precision' : [],
        'S_recall' : [],
        'S_f1-score' : [],
        'S_support' : [],
        'C_precision' : [],
        'C_recall' : [],
        'C_f1-score' : [],
        'C_support' : [],
        'macro_precision' : [],
        'macro_recall' : [],
        'macro_f1-score' : [],
        'macro_support': [],
        'LC_precision' : [],
        'LC_recall' : [],
        'LC_f1-score' : [],
        'LC_support' : [],
        'IMD_N_precision' : [],
        'IMD_N_recall' : [],
        'IMD_N_f1-score' : [],
        'IMD_N_support' : [],
        'IMD_S_precision' : [],
        'IMD_S_recall' : [],
        'IMD_S_f1-score' : [],
        'IMD_S_support' : [],
        'IMD_C_precision' : [],
        'IMD_C_recall' : [],
        'IMD_C_f1-score' : [],
        'IMD_C_support' : [],
        'IMD_macro_precision' : [],
        'IMD_macro_recall' : [],
        'IMD_macro_f1-score' : [],
        'IMD_macro_support': [],

    })

i=0
for argname in argname_list:
    
    pkl_dict = {} # For save
    ## ----------------------------------------------------------------
    ## Dataset - source
    y_source = y_source_all[argname]
    idx_source = y_source.index.tolist()
    Scaler_source = Scaler_source_all[argname]

    y_target_NSC = data_dict_target['Outputs_NSC']
    y_target_LC = data_dict_target['Outputs_LC']
    x_target = data_dict_target['Inputs']
    idx_target_NSC = y_target_NSC.index.tolist()
    idx_target_LC = y_target_LC.index.tolist()
    
    
    # Loop combination
    n_samples_source_list = [0]+(np.floor(10**np.linspace(2, np.log10(len(idx_source)), 10))).astype(np.int32).tolist()

    combinations = list(itertools.product(
        n_seed_list,
        n_samples_source_list
    ))
    
    ## ----------------------------------------------------------------
    ## Directories
    result_path = f'./Result/{argname}'
    model_path = f'./Result/{argname}/Target_Class_Model'
    fun_MakeDir([result_path, model_path])
    
    for n_seed, n_samples_source in combinations:
        print(argname)

        loop_id = f'Seed{n_seed}NumSource{n_samples_source}'
        print(loop_id)
        
        ## ----------------------------------------------------------------
        ## Model definition
        fix_seed(n_seed)
        model = MyModel(
            input_dim = x_target.shape[1], 
            layers_dict = {
                'fc1': 512, 
                'fc2': 256,
                'fc3': 128,
                'fc4': 64,
                'fc5': 32,
                'fc6': 1
            }
        )
        model.initialize_weights(seed=n_seed)
        model.to(MyDevice)

        ### Training parameters
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam
        learning_rate_source = 1e-3
        learning_rate_target = {
            'fc1' : 1e-3,
            'fc2' : 1e-3,
            'fc3' : 1e-3,
            'fc4' : 1e-3,
            'fc5' : 1e-3,
            'fc6' : 1e-3
        }
        n_epochs = 1000


        ### Early stopping parameters
        patience = 30
        loss_th_source = None
        loss_th_target = None
        best_loss = None
        epochs_no_improve = 0
        
        ## ----------------------------------------------------------------
        ## Data splitting
        ### Target
        idx_trainvalcal_target_NSC, idx_test_target_NSC = train_test_split(idx_target_NSC, random_state=n_seed, test_size=0.2)
        idx_train_target_NSC, idx_valcal_target_NSC = train_test_split(idx_trainvalcal_target_NSC, random_state=n_seed, test_size=2/8)
        idx_val_target_NSC, idx_cal_target_NSC = train_test_split(idx_valcal_target_NSC, random_state=n_seed, test_size=1/2)
        idx_trainvalcal_target_LC, idx_test_target_LC = train_test_split(idx_target_LC, random_state=n_seed, test_size=0.2)
        idx_train_target_LC, idx_valcal_target_LC = train_test_split(idx_trainvalcal_target_LC, random_state=n_seed, test_size=2/8)
        idx_val_target_LC, idx_cal_target_LC = train_test_split(idx_valcal_target_LC, random_state=n_seed, test_size=1/2)
            
        ## ----------------------------------------------------------------
        ## Create dataset and dataloader
        ### Dataset NSC
        train_x_NSC_tensor = torch.Tensor(x_target.loc[idx_train_target_NSC].values).float()
        train_y_NSC_tensor = torch.Tensor(y_target_NSC.loc[idx_train_target_NSC].values).float()
        train_tensor_NSC = TensorDataset(train_x_NSC_tensor, train_y_NSC_tensor)

        val_x_NSC_tensor = torch.Tensor(x_target.loc[idx_val_target_NSC].values).float()
        val_y_NSC_tensor = torch.Tensor(y_target_NSC.loc[idx_val_target_NSC].values).float()
        val_tensor_NSC = TensorDataset(val_x_NSC_tensor, val_y_NSC_tensor)
        #Data loader
        train_loader_NSC = DataLoader(train_tensor_NSC, batch_size = 32, shuffle = True)
        val_loader_NSC = DataLoader(val_tensor_NSC, batch_size = 32, shuffle = False)
        ### Dataset LC
        train_x_LC_tensor = torch.Tensor(x_target.loc[idx_train_target_LC].values).float()
        train_y_LC_tensor = torch.Tensor(y_target_LC.loc[idx_train_target_LC].astype(bool).values.reshape(-1,1)).float()
        train_tensor_LC = TensorDataset(train_x_LC_tensor, train_y_LC_tensor)

        val_x_LC_tensor = torch.Tensor(x_target.loc[idx_val_target_LC].values).float()
        val_y_LC_tensor = torch.Tensor(y_target_LC.loc[idx_val_target_LC].astype(bool).values.reshape(-1,1)).float()
        val_tensor_LC = TensorDataset(val_x_LC_tensor, val_y_LC_tensor)
        #Data loader
        train_loader_LC = DataLoader(train_tensor_LC, batch_size = 32, shuffle = True)
        val_loader_LC = DataLoader(val_tensor_LC, batch_size = 32, shuffle = False)


        max_len = max(len(train_loader_NSC), len(train_loader_LC))

        train_loader_NSC_cycle = cycle(train_loader_NSC)
        train_loader_LC_cycle = cycle(train_loader_LC)

        dim = x_target.shape[1]
        inter_dim_1 = 512
        inter_dim_2 = 256
        inter_dim_3 = 128
        inter_dim_4 = 64
        inter_dim_5 = 32
        
        
        class Model(nn.Module):
            def __init__(self):
                super().__init__()
                self.linear_1 = nn.Linear(dim, inter_dim_1)
                self.linear_2 = nn.Linear(inter_dim_1, inter_dim_2)
                self.linear_3 = nn.Linear(inter_dim_2, inter_dim_3)
                self.linear_4 = nn.Linear(inter_dim_3, inter_dim_4)
                
                self.linear_5_NSC = nn.Linear(inter_dim_4, inter_dim_5)
                self.linear_5_LC = nn.Linear(inter_dim_4, inter_dim_5)
                self.linear_out_NSC = nn.Linear(inter_dim_5, 3)
                self.linear_out_LC = nn.Linear(inter_dim_5, 1)
                
            def forward(self, x):
                x = F.relu(self.linear_1(x))
                x = F.relu(self.linear_2(x))
                x = F.relu(self.linear_3(x))
                x = F.relu(self.linear_4(x))
                x_NSC = F.relu(self.linear_5_NSC(x))
                out_NSC = self.linear_out_NSC(x_NSC)
                x_LC = F.relu(self.linear_5_LC(x))
                out_LC = self.linear_out_LC(x_LC)
                
                return out_NSC, out_LC
            
        model = Model()

        if n_samples_source > 0:
        ## ----------------------------------------------------------------
        #Load
            param_tl = torch.load(f'./Result/{argname}/Source_Model/SourceModelWeight_{loop_id}.pth')
            param = model.state_dict()
            param['linear_1.weight'] = param_tl['layers.fc1.weight']
            param['linear_1.bias'] = param_tl['layers.fc1.bias']
            param['linear_2.weight'] = param_tl['layers.fc2.weight']
            param['linear_2.bias'] = param_tl['layers.fc2.bias']
            param['linear_3.weight'] = param_tl['layers.fc3.weight']
            param['linear_3.bias'] = param_tl['layers.fc3.bias']
            param['linear_4.weight'] = param_tl['layers.fc4.weight']
            param['linear_4.bias'] = param_tl['layers.fc4.bias']

            model.load_state_dict(param)
            
            # Initialize head layer
            nn.init.kaiming_uniform_(model.linear_5_NSC.weight)
            nn.init.constant_(model.linear_5_NSC.bias, 0.01)
            nn.init.kaiming_uniform_(model.linear_out_NSC.weight)
            nn.init.constant_(model.linear_out_NSC.bias, 0.01)
            nn.init.kaiming_uniform_(model.linear_5_LC.weight)
            nn.init.constant_(model.linear_5_LC.bias, 0.01)
            nn.init.kaiming_uniform_(model.linear_out_LC.weight)
            nn.init.constant_(model.linear_out_LC.bias, 0.01)
        else:
            nn.init.kaiming_uniform_(model.linear_1.weight)
            nn.init.constant_(model.linear_1.bias, 0.01)
            nn.init.kaiming_uniform_(model.linear_2.weight)
            nn.init.constant_(model.linear_2.bias, 0.01)
            nn.init.kaiming_uniform_(model.linear_3.weight)
            nn.init.constant_(model.linear_3.bias, 0.01)
            nn.init.kaiming_uniform_(model.linear_4.weight)
            nn.init.constant_(model.linear_4.bias, 0.01)
            nn.init.kaiming_uniform_(model.linear_5_NSC.weight)
            nn.init.constant_(model.linear_5_NSC.bias, 0.01)
            nn.init.kaiming_uniform_(model.linear_out_NSC.weight)
            nn.init.constant_(model.linear_out_NSC.bias, 0.01)
            nn.init.kaiming_uniform_(model.linear_5_LC.weight)
            nn.init.constant_(model.linear_5_LC.bias, 0.01)
            nn.init.kaiming_uniform_(model.linear_out_LC.weight)
            nn.init.constant_(model.linear_out_LC.bias, 0.01)
        


        # 最適化関数の設定
        optimizer = torch.optim.Adam(model.parameters())
        pos_weight = 2
        patience = 5
        n_epochs = 1000
        delta = 0.2
        best_loss = None
        
        # 損失関数の設定
        #pos_weight=torch.Tensor([weight])
        loss_function_NSC =  nn.BCEWithLogitsLoss(pos_weight=torch.Tensor([pos_weight]))
        loss_function_LC =  nn.BCEWithLogitsLoss(pos_weight=torch.Tensor([pos_weight]))

        model.train()
        losses = []

        for epoch in list(range(n_epochs)):
            
            train_loss = 0
            
            # バッチごとの処理
            for _ in range(max_len):
                batch_NSC = next(train_loader_NSC_cycle)
                batch_LC = next(train_loader_LC_cycle)
                batch_X_NSC, batch_y_NSC = batch_NSC[0], batch_NSC[1]
                batch_X_LC, batch_y_LC = batch_LC[0], batch_LC[1]
                
                # トレーニングデータに対する誤差の算出
                batch_pred_y_NSC = model(batch_X_NSC)[0]
                batch_pred_y_LC = model(batch_X_LC)[1]
                
                batch_loss_NSC = loss_function_NSC(batch_pred_y_NSC, batch_y_NSC)
                batch_loss_LC = loss_function_LC(batch_pred_y_LC, batch_y_LC)
                
                batch_loss = (1-delta)*batch_loss_NSC + delta*batch_loss_LC
                
                # 誤差逆伝播と重み更新
                optimizer.zero_grad()
                batch_loss.backward()
                optimizer.step()
                
                train_loss += batch_loss.item()
                
            losses.append(train_loss)
            
            model.eval()
            with torch.no_grad():
                val_loss = 0
                pred_y_NSC = torch.Tensor()
                pred_y_LC = torch.Tensor()

                for _ in range(max_len):
                    batch_NSC = next(train_loader_NSC_cycle)
                    batch_LC = next(train_loader_LC_cycle)
                    batch_X_NSC, batch_y_NSC = batch_NSC[0], batch_NSC[1]
                    batch_X_LC, batch_y_LC = batch_LC[0], batch_LC[1]
                    
                    # トレーニングデータに対する誤差の算出
                    batch_pred_y_NSC = model(batch_X_NSC)[0]
                    batch_pred_y_LC = model(batch_X_LC)[1]
                    
                    batch_loss_NSC = loss_function_NSC(batch_pred_y_NSC, batch_y_NSC)
                    batch_loss_LC = loss_function_LC(batch_pred_y_LC, batch_y_LC)
                    
                    batch_loss = (1-delta)*batch_loss_NSC + delta*batch_loss_LC

                    val_loss += batch_loss.item()
                    
            if (epoch == 0) or ((epoch+1) % 10 == 0):
                print(f'Epoch {epoch+1}: train loss = {train_loss}, val loss = {val_loss}')
            
            if patience is not None:
                # Early stopping
                if best_loss is None:
                    best_loss = val_loss
                    best_weights = copy.deepcopy(model.state_dict())
                elif val_loss < best_loss:
                    best_loss = val_loss
                    epochs_no_improve = 0
                    best_weights = copy.deepcopy(model.state_dict())
                else:
                    epochs_no_improve += 1
                    if epochs_no_improve == patience:
                        # clear_output()
                        print('Early stopping!')
                        print('======================')
                        break
            if epoch==(n_epochs-1):
                # clear_output()
                print('Reach max epochs')
                print('======================')
                
            if patience is not None:
                model.load_state_dict(best_weights)
                
        torch.save(model.state_dict(), f'{model_path}/TargetModelWeight_{loop_id}.pth')
        
        
        
        #Evaluate model    
        pred_y_NSC = (model(torch.Tensor(x_target.loc[idx_test_target_NSC].values).float())[0] > 0).detach().numpy().astype('int32')
        pred_y_LC = (model(torch.Tensor(x_target.loc[idx_test_target_LC].values).float())[1] > 0).detach().numpy().astype('int32')

        cr_NSC = classification_report(y_target_NSC.loc[idx_test_target_NSC], pred_y_NSC,
                            target_names=['N','S','C'],
                            output_dict=True, zero_division=0)
        
        cr_LC = {'precision': precision_score(y_target_LC.loc[idx_test_target_LC].astype(int), pred_y_LC, zero_division=0),
                'recall': recall_score(y_target_LC.loc[idx_test_target_LC].astype(int), pred_y_LC, zero_division=0),
                'f1-score': f1_score(y_target_LC.loc[idx_test_target_LC].astype(int), pred_y_LC, zero_division=0),
                'support': len(pred_y_LC)
                    }
        
        test_IMD_idx = list(set(idx_test_target_NSC) & set(IMD_idx))
        df_pred_y_NSC = pd.DataFrame(pred_y_NSC, columns=['N','S','C'], index=idx_test_target_NSC)

        cr_IMD = classification_report(y_target_NSC.loc[test_IMD_idx], df_pred_y_NSC.loc[test_IMD_idx],
                            target_names=['N','S','C'],
                            output_dict=True,
                            zero_division=0)
        

        
        df_results.loc[len(df_results)] = [argname, n_seed,
                                           n_samples_source,
                                            cr_NSC['N']['precision'],
                                            cr_NSC['N']['recall'],
                                            cr_NSC['N']['f1-score'],
                                            cr_NSC['N']['support'],
                                            cr_NSC['S']['precision'],
                                            cr_NSC['S']['recall'],
                                            cr_NSC['S']['f1-score'],
                                            cr_NSC['S']['support'],
                                            cr_NSC['C']['precision'],
                                            cr_NSC['C']['recall'],
                                            cr_NSC['C']['f1-score'],
                                            cr_NSC['C']['support'],
                                            cr_NSC['macro avg']['precision'],
                                            cr_NSC['macro avg']['recall'],
                                            cr_NSC['macro avg']['f1-score'],
                                            cr_NSC['macro avg']['support'],
                                            cr_LC['precision'],
                                            cr_LC['recall'],
                                            cr_LC['f1-score'],
                                            cr_LC['support'],
                                            cr_IMD['N']['precision'],
                                            cr_IMD['N']['recall'],
                                            cr_IMD['N']['f1-score'],
                                            cr_IMD['N']['support'],
                                            cr_IMD['S']['precision'],
                                            cr_IMD['S']['recall'],
                                            cr_IMD['S']['f1-score'],
                                            cr_IMD['S']['support'],
                                            cr_IMD['C']['precision'],
                                            cr_IMD['C']['recall'],
                                            cr_IMD['C']['f1-score'],
                                            cr_IMD['C']['support'],
                                            cr_IMD['macro avg']['precision'],
                                            cr_IMD['macro avg']['recall'],
                                            cr_IMD['macro avg']['f1-score'],
                                            cr_IMD['macro avg']['support'],
                                            ]
    
        df_results.to_csv(f'./Result/Class_scale.csv')
        
        
        train_pred_y_NSC = pd.DataFrame(sigmoid(model(torch.Tensor(x_target.loc[idx_train_target_NSC].values).float())[0].detach().numpy()), columns=['N','S','C'], index=idx_train_target_NSC)
        cal_pred_y_NSC = pd.DataFrame(sigmoid(model(torch.Tensor(x_target.loc[idx_cal_target_NSC].values).float())[0].detach().numpy()), columns=['N','S','C'], index=idx_cal_target_NSC)        
        pkl_dict[f'{loop_id}'] = {
                                        'train_idx_NSC': idx_train_target_NSC,
                                        'cal_idx_NSC':idx_cal_target_NSC,
                                        'test_idx_NSC': idx_test_target_NSC,
                                        'train_x_NSC':x_target.loc[idx_train_target_NSC],
                                        'train_y_NSC':y_target_NSC.loc[idx_train_target_NSC],
                                        'train_pred_y_NSC': train_pred_y_NSC,
                                        'cal_x_NSC': x_target.loc[idx_cal_target_NSC],
                                        'cal_y_NSC': y_target_NSC.loc[idx_cal_target_NSC],
                                        'cal_pred_y_NSC': cal_pred_y_NSC,
                                        'model':model.state_dict(),
                                        'scaler_x':data_dict_target['Scaler_inputs']}
        with open(f'{model_path}/all.pickle', 'wb') as f:
            pickle.dump(pkl_dict, f)
        f.close()
    

density
Seed0NumSource0
Epoch 1: train loss = 6.755792677402496, val loss = 4.3061690628528595
Epoch 10: train loss = 1.6816836297512054, val loss = 1.47226994484663
Epoch 20: train loss = 1.2579423263669014, val loss = 1.2366379722952843
Epoch 30: train loss = 0.9463855773210526, val loss = 0.8266210183501244
Early stopping!
density
Seed0NumSource100
Epoch 1: train loss = 7.0974313616752625, val loss = 4.255415767431259
Epoch 10: train loss = 1.7827038541436195, val loss = 1.5590504854917526
Epoch 20: train loss = 1.0746162943542004, val loss = 0.9603549987077713
Early stopping!
density
Seed0NumSource205
Epoch 1: train loss = 7.252660840749741, val loss = 4.507137447595596
Epoch 10: train loss = 1.4690120369195938, val loss = 1.3840512931346893
Epoch 20: train loss = 1.163663111627102, val loss = 0.9746025986969471
Epoch 30: train loss = 1.002103827893734, val loss = 0.8467729724943638
Early stopping!
density
Seed0NumSource420
Epoch 1: train loss = 7.033388167619705, val loss = 4.4153